In [64]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import RandomizedSearchCV

warnings.filterwarnings("ignore")

In [65]:
df = pd.read_csv("data/raw_data.csv")

In [66]:
df.head(5)

,customer_id,region,subscription_plan,contract_type,tenure_months,monthly_logins,customer_satisfaction_score,industry,company_size,years_in_business,...,payment_failures_last_12m,email_open_rate,webinar_attendance_count,monthly_recurring_revenue,lifetime_value,signup_date,last_login_date,customer_segment,acquisition_channel,churn
0,CF00001,North West,Professional,Monthly,3,42.0,2.0,Technology,Medium,30,...,0,96.0,3,92.24,276.72,2024-12-27,2025-06-20,SMB,LinkedIn,Yes
1,CF00002,East Midlands,Starter,Monthly,1,45.0,7.0,Retail,Small,15,...,2,33.0,1,75.62,75.62,2022-11-11,2023-03-06,SMB,LinkedIn,No
2,CF00003,Scotland,Starter,Monthly,14,59.0,4.0,Healthcare,Small,3,...,2,90.0,7,72.83,1019.62,2023-05-01,2023-10-18,SMB,Partner,Yes
3,CF00004,West Midlands,Professional,Annual,16,48.0,7.0,Logistics,Small,1,...,2,79.0,10,74.76,1196.16,2025-04-01,2025-05-17,Mid-Market,Organic Search,No
4,CF00005,North West,Professional,Monthly,1,50.0,10.0,Education,Micro,15,...,2,0.0,1,29.92,29.92,2021-09-04,2022-02-05,Enterprise,Partner,No


In [67]:
# Rule-based cleaning
df = df.drop_duplicates()

df["region"] = (
    df["region"]
      .str.strip()
      .str.title()
)

df["customer_satisfaction_score"] = (
    df["customer_satisfaction_score"]
      .clip(1, 10)
)

df["tenure_months"] = (
    df["tenure_months"]
      .abs()
)


In [68]:
# Independent variables
X = df.drop(
    columns=[
        "customer_id",
        "signup_date",
        "last_login_date",
        "churn"
    ]
)

# Target variable
y = df["churn"]

In [69]:
X.head(5)

,region,subscription_plan,contract_type,tenure_months,monthly_logins,customer_satisfaction_score,industry,company_size,years_in_business,monthly_subscription_fee_gbp,...,integrations_enabled,support_tickets_last_6m,avg_resolution_time_hours,payment_failures_last_12m,email_open_rate,webinar_attendance_count,monthly_recurring_revenue,lifetime_value,customer_segment,acquisition_channel
0,North West,Professional,Monthly,3,42.0,2.0,Technology,Medium,30,79,...,5,1,7,0,96.0,3,92.24,276.72,SMB,LinkedIn
1,East Midlands,Starter,Monthly,1,45.0,7.0,Retail,Small,15,79,...,0,1,57,2,33.0,1,75.62,75.62,SMB,LinkedIn
2,Scotland,Starter,Monthly,14,59.0,4.0,Healthcare,Small,3,79,...,17,2,34,2,90.0,7,72.83,1019.62,SMB,Partner
3,West Midlands,Professional,Annual,16,48.0,7.0,Logistics,Small,1,79,...,9,3,27,2,79.0,10,74.76,1196.16,Mid-Market,Organic Search
4,North West,Professional,Monthly,1,50.0,10.0,Education,Micro,15,29,...,10,3,45,2,0.0,1,29.92,29.92,Enterprise,Partner


In [70]:
X.shape

(15029, 26)

In [71]:
y.head(5)

0    Yes
1     No
2    Yes
3     No
4     No
Name: churn, dtype: object

In [72]:
y.shape

(15029,)

Splitting Data for Train and Test

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Categorical columns and Numerical Columns 

In [74]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features:", categorical_cols)
print("Numerical Features:", numerical_cols)

Categorical Features: ['region', 'subscription_plan', 'contract_type', 'industry', 'company_size', 'customer_segment', 'acquisition_channel']
Numerical Features: ['tenure_months', 'monthly_logins', 'customer_satisfaction_score', 'years_in_business', 'monthly_subscription_fee_gbp', 'discount_percentage', 'employee_count', 'active_users', 'feature_usage_score', 'avg_session_duration_minutes', 'projects_created', 'integrations_enabled', 'support_tickets_last_6m', 'avg_resolution_time_hours', 'payment_failures_last_12m', 'email_open_rate', 'webinar_attendance_count', 'monthly_recurring_revenue', 'lifetime_value']


In [75]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [76]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [77]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

Encode Target Variable

Since your target is "Yes" and "No":

In [78]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)

y_test = label_encoder.transform(y_test)

This converts:

No  -> 0

Yes -> 1

In [79]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(12023, 53)
(3006, 53)
(12023,)
(3006,)


Smote

In [80]:
smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

In [81]:
print(pd.Series(y_train).value_counts())

0    8888
1    8888
Name: count, dtype: int64


# Model Training

In [82]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    "CatBoost": CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}

In [83]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    roc_auc = roc_auc_score(
        y_test,
        model.predict_proba(X_test)[:, 1]
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ])

In [84]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
)

results_df.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
4,CatBoost,0.880572,0.710605,0.914541,0.799777,0.921488
3,XGBoost,0.876580,0.717140,0.869898,0.786167,0.924718
2,Random Forest,0.868929,0.709227,0.843112,0.770396,0.919392
1,Decision Tree,0.830339,0.661176,0.716837,0.687882,0.793612
0,Logistic Regression,0.764138,0.533572,0.760204,0.627038,0.839037


Hyperpara Meter Tuning 

Hyperparameter Tuning - Random Forest

In [99]:
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=rf_params,
    n_iter=10,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [5, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,n_iter,10
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [103]:
rf_best = rf_search.best_estimator_

rf_pred = rf_best.predict(X_test)
rf_prob = rf_best.predict_proba(X_test)[:,1]

rf_results = {
    "Model":"Random Forest (Tuned)",
    "Accuracy":accuracy_score(y_test,rf_pred),
    "Precision":precision_score(y_test,rf_pred),
    "Recall":recall_score(y_test,rf_pred),
    "F1 Score":f1_score(y_test,rf_pred),
    "ROC-AUC":roc_auc_score(y_test,rf_prob)
}

rf_results

{'Model': 'Random Forest (Tuned)',
 'Accuracy': 0.8819028609447771,
 'Precision': 0.7173252279635258,
 'Recall': 0.9030612244897959,
 'F1 Score': 0.7995482778091474,
 'ROC-AUC': 0.9199614476753798}

Hyperparameter Tuning - XGBoost

In [104]:
xgb_params = {
    "n_estimators":[100,200,300],
    "learning_rate":[0.01,0.05,0.1],
    "max_depth":[3,5,7],
    "subsample":[0.8,1],
    "colsample_bytree":[0.8,1]
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_distributions=xgb_params,
    n_iter=10,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train,y_train)

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.8, 1], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], ...}"
,n_iter,10
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [110]:
xgb_best = xgb_search.best_estimator_

xgb_pred = xgb_best.predict(X_test)
xgb_prob = xgb_best.predict_proba(X_test)[:,1]

xgb_results = {
    "Model":"XGBoost (Tuned)",
    "Accuracy":accuracy_score(y_test,xgb_pred),
    "Precision":precision_score(y_test,xgb_pred),
    "Recall":recall_score(y_test,xgb_pred),
    "F1 Score":f1_score(y_test,xgb_pred),
    "ROC-AUC":roc_auc_score(y_test,xgb_prob)
}

xgb_results

{'Model': 'XGBoost (Tuned)',
 'Accuracy': 0.8862275449101796,
 'Precision': 0.7120921305182342,
 'Recall': 0.9464285714285714,
 'F1 Score': 0.8127053669222344,
 'ROC-AUC': 0.922517634416503}

Hyperparameter Tuning - CatBoost

In [106]:
cat_params = {
    "depth":[4,6,8,10],
    "learning_rate":[0.01,0.05,0.1],
    "iterations":[100,200,300],
    "l2_leaf_reg":[1,3,5,7]
}

cat_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        random_state=42,
        verbose=0
    ),
    param_distributions=cat_params,
    n_iter=10,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

cat_search.fit(X_train,y_train)

,estimator,"CatBoostClass...42, verbose=0)"
,param_distributions,"{'depth': [4, 6, ...], 'iterations': [100, 200, ...], 'l2_leaf_reg': [1, 3, ...], 'learning_rate': [0.01, 0.05, ...]}"
,n_iter,10
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [107]:
cat_best = cat_search.best_estimator_

cat_pred = cat_best.predict(X_test)
cat_prob = cat_best.predict_proba(X_test)[:,1]

cat_results = {
    "Model":"CatBoost (Tuned)",
    "Accuracy":accuracy_score(y_test,cat_pred),
    "Precision":precision_score(y_test,cat_pred),
    "Recall":recall_score(y_test,cat_pred),
    "F1 Score":f1_score(y_test,cat_pred),
    "ROC-AUC":roc_auc_score(y_test,cat_prob)
}

Compare Tuned Models

In [108]:
tuned_results = pd.DataFrame([
    rf_results,
    xgb_results,
    cat_results
])

tuned_results.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,XGBoost (Tuned),0.886228,0.712092,0.946429,0.812705,0.922518
2,CatBoost (Tuned),0.885895,0.711816,0.945153,0.812055,0.918130
0,Random Forest (Tuned),0.881903,0.717325,0.903061,0.799548,0.919961


In [120]:
print("Best Parameters:")
print(xgb_search.best_params_)

Best Parameters:
{'subsample': 1, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
